In [1]:
import time
import torch

import pandas as pd
import torch.nn as nn

from torch_geometric.data import Data


from src.data.load_raw import load_raw_data
from src.data.preprocess import (
    extract_xy,
    merge_features_and_labels,
    standardize_splits,
)
from src.data.split import split_labeled_transactions
from src.evaluation.metrics import calculate_binary_metrics
from src.evaluation.threshold import find_best_f1_threshold
from sklearn.model_selection import TimeSeriesSplit
from src.models.classic import (
    build_dummy_classifier,
    build_logistic_regression,
    build_random_forest,
    build_xgboost,
)

import optuna
from sklearn.model_selection import cross_val_score
import numpy as np

import warnings
from sklearn.exceptions import ConvergenceWarning
from src.models.gcn import build_gcn

warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.linear_model")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.linear_model")

warnings.filterwarnings("ignore", category=ConvergenceWarning)

C:\Users\pansm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def prepare_gcn_data(splits, x_train_scaled, x_val_scaled, x_test_scaled, y_train, y_val, y_test, edges_df):
    x = torch.cat([
        torch.tensor(x_train_scaled, dtype=torch.float),
        torch.tensor(x_val_scaled, dtype=torch.float),
        torch.tensor(x_test_scaled, dtype=torch.float)
    ], dim=0)

    y = torch.cat([
        torch.tensor(y_train.values, dtype=torch.long),
        torch.tensor(y_val.values, dtype=torch.long),
        torch.tensor(y_test.values, dtype=torch.long)
    ], dim=0)

    num_train = len(x_train_scaled)
    num_val = len(x_val_scaled)
    num_test = len(x_test_scaled)
    total_nodes = num_train + num_val + num_test

    train_mask = torch.zeros(total_nodes, dtype=torch.bool)
    val_mask = torch.zeros(total_nodes, dtype=torch.bool)
    test_mask = torch.zeros(total_nodes, dtype=torch.bool)

    train_mask[:num_train] = True
    val_mask[num_train : num_train + num_val] = True
    test_mask[num_train + num_val :] = True

    ordered_tx_ids = pd.concat([
        splits.train["tx_id"],
        splits.validation["tx_id"],
        splits.test["tx_id"]
    ]).values

    node_mapping = {tx_id: i for i, tx_id in enumerate(ordered_tx_ids)}

    edges_df["source_mapped"] = edges_df["source_tx_id"].map(node_mapping)
    edges_df["target_mapped"] = edges_df["target_tx_id"].map(node_mapping)

    edges_mapped = edges_df.dropna(subset=["source_mapped", "target_mapped"])

    edge_index = torch.tensor([
        edges_mapped["source_mapped"].values,
        edges_mapped["target_mapped"].values
    ], dtype=torch.long)

    graph_data = Data(x=x, y=y, edge_index=edge_index)

    return graph_data, train_mask, val_mask, test_mask

In [3]:
def run_gcn_experiment(model_name, model, data, train_mask, val_mask, test_mask, epochs=200, lr=0.001):
    start_time = time.perf_counter()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    pos_weight = torch.tensor([0.7 / 0.3]).to(model.device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()

        out, _ = model(data.x, data.edge_index)

        loss = criterion(out[train_mask].squeeze(), data.y[train_mask].float())

        loss.backward()
        optimizer.step()

        if (epoch + 1) % 50 == 0:
            print(f"GCN Epoch {epoch+1}/{epochs} | Train Loss: {loss.item():.4f}")

    training_time = time.perf_counter() - start_time

    model.eval()
    with torch.no_grad():
        logits, final_embeddings = model(data.x, data.edge_index)
        probabilities = torch.sigmoid(logits).squeeze().cpu().numpy()

    val_scores = probabilities[val_mask.cpu().numpy()]
    y_val_true = data.y[val_mask].cpu().numpy()
    threshold = find_best_f1_threshold(y_val_true, val_scores)

    test_scores = probabilities[test_mask.cpu().numpy()]
    y_test_true = data.y[test_mask].cpu().numpy()

    metrics = calculate_binary_metrics(y_test_true, test_scores, threshold=threshold)

    metrics.update({
        "model": model_name,
        "feature_set": "all",
        "training_time_seconds": training_time,
    })

    return model, metrics, final_embeddings

In [4]:
features_df, classes_df, edges_df = load_raw_data()

transactions_df = merge_features_and_labels(
    features_df,
    classes_df,
)

splits = split_labeled_transactions(transactions_df)

x_train, y_train = extract_xy(
    splits.train,
    feature_set="local",
)

x_validation, y_validation = extract_xy(
    splits.validation,
    feature_set="local",
)

x_test, y_test = extract_xy(
    splits.test,
    feature_set="local",
)

(
    scaler,
    x_train_scaled,
    x_validation_scaled,
    x_test_scaled,
) = standardize_splits(
    x_train,
    x_validation,
    x_test,
)

x_all_train, y_all_train = extract_xy(
    splits.train,
    feature_set="all",
)

x_all_validation, y_all_validation = extract_xy(
    splits.validation,
    feature_set="all",
)

x_all_test, y_all_test = extract_xy(
    splits.test,
    feature_set="all",
)

(
    all_scaler,
    x_all_train_scaled,
    x_all_validation_scaled,
    x_all_test_scaled,
) = standardize_splits(
    x_all_train,
    x_all_validation,
    x_all_test,
)

graph_data, train_mask, val_mask, test_mask = prepare_gcn_data(
    splits=splits,
    x_train_scaled=x_train_scaled,
    x_val_scaled=x_validation_scaled,
    x_test_scaled=x_test_scaled,
    y_train=y_train,
    y_val=y_validation,
    y_test=y_test,
    edges_df=edges_df
)


gcn_all_model, gcn_metrics, gcn_embeddings = run_gcn_experiment(
    model_name="GCN",
    model=build_gcn(in_features=94, hidden_size=100),
    data=graph_data,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

embeddings_all = (
    torch.relu(gcn_embeddings)
    .detach()
    .cpu()
    .numpy()
)

train_embeddings = embeddings_all[train_mask.cpu().numpy()]
validation_embeddings = embeddings_all[val_mask.cpu().numpy()]
test_embeddings = embeddings_all[test_mask.cpu().numpy()]

def append_gcn_embeddings(x_df, embeddings):
    embedding_columns = [
        f"gcn_embedding_{i + 1}"
        for i in range(embeddings.shape[1])
    ]

    embeddings_df = pd.DataFrame(
        embeddings,
        columns=embedding_columns,
        index=x_df.index,
    )

    return pd.concat(
        [x_df.copy(), embeddings_df],
        axis=1,
    )


x_all_gcn_train = append_gcn_embeddings(
    x_all_train,
    train_embeddings,
)

x_all_gcn_validation = append_gcn_embeddings(
    x_all_validation,
    validation_embeddings,
)

x_all_gcn_test = append_gcn_embeddings(
    x_all_test,
    test_embeddings,
)

(
    all_gcn_scaler,
    x_all_gcn_train_scaled,
    x_all_gcn_validation_scaled,
    x_all_gcn_test_scaled,
) = standardize_splits(
    x_all_gcn_train,
    x_all_gcn_validation,
    x_all_gcn_test,
)

C:\Users\pansm\AppData\Local\Temp\ipykernel_8636\3041131662.py:40: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  edge_index = torch.tensor([


GCN Epoch 50/200 | Train Loss: 0.3524
GCN Epoch 100/200 | Train Loss: 0.2830
GCN Epoch 150/200 | Train Loss: 0.2482
GCN Epoch 200/200 | Train Loss: 0.2224


In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def objective1(trial, X, y):
    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear", "saga"])

    if solver == "lbfgs":
        reg_type = trial.suggest_categorical("reg_lbfgs", ["l2", "none"])
        C = np.inf if reg_type == "none" else trial.suggest_float("C_lbfgs", 1e-4, 1e4, log=True)
        l1_ratio = None
    elif solver == "liblinear":
        reg_type = trial.suggest_categorical("reg_lib", ["l1", "l2"])
        C = trial.suggest_float("C_lib", 1e-4, 1e4, log=True)
        l1_ratio = None
    else: # saga
        reg_type = trial.suggest_categorical("reg_saga", ["l1", "l2", "elasticnet", "none"])
        if reg_type == "none":
            C = np.inf
            l1_ratio = None
        else:
            C = trial.suggest_float("C_saga", 1e-4, 1e4, log=True)
            if reg_type == "l1":
                l1_ratio = 1.0
            elif reg_type == "l2":
                l1_ratio = 0.0
            else:
                l1_ratio = trial.suggest_float("l1_ratio_saga", 0.01, 0.99)

    class_weight = trial.suggest_categorical("class_weight", ["balanced", None])
    max_iter = trial.suggest_int("max_iter", 500, 2000, step=500)

    model = build_logistic_regression(
        solver=solver,
        C=C,
        l1_ratio=l1_ratio,
        max_iter=max_iter,
        class_weight=class_weight
    )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", model)
    ])

    tscv = TimeSeriesSplit(n_splits=5)

    scores = cross_val_score(
        pipeline, X, y, cv=tscv, scoring="f1", n_jobs=-1
    )

    return np.mean(scores[-3:])


study = optuna.create_study(direction="maximize", study_name="Logistic_Regression_Tuning")

study.optimize(lambda trial: objective1(trial, x_all_train, y_all_train), n_trials=50, show_progress_bar=True)

print("\n--- Optimization Finished ---")
print(f"Best Trial Score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# --- Optimization Finished ---
# Best Trial Score: 0.7941
# Best Parameters:
#   solver: lbfgs
#   reg_lbfgs: none
#   class_weight: None
#   max_iter: 1000

[I 2026-06-15 22:53:53,487] A new study created in memory with name: Logistic_Regression_Tuning
Best trial: 0. Best value: 0.751241:   2%|▏         | 1/50 [00:24<20:20, 24.90s/it]

[I 2026-06-15 22:54:18,392] Trial 0 finished with value: 0.7512407499106651 and parameters: {'solver': 'liblinear', 'reg_lib': 'l2', 'C_lib': 52.20661784953828, 'class_weight': None, 'max_iter': 500}. Best is trial 0 with value: 0.7512407499106651.


Best trial: 0. Best value: 0.751241:   4%|▍         | 2/50 [00:29<10:25, 13.03s/it]

[I 2026-06-15 22:54:23,114] Trial 1 finished with value: 0.6310226188520699 and parameters: {'solver': 'lbfgs', 'reg_lbfgs': 'none', 'class_weight': 'balanced', 'max_iter': 2000}. Best is trial 0 with value: 0.7512407499106651.


Best trial: 0. Best value: 0.751241:   6%|▌         | 3/50 [00:33<06:45,  8.63s/it]

[I 2026-06-15 22:54:26,501] Trial 2 finished with value: 0.569024691931392 and parameters: {'solver': 'lbfgs', 'reg_lbfgs': 'l2', 'C_lbfgs': 1.4067456363133881, 'class_weight': 'balanced', 'max_iter': 1500}. Best is trial 0 with value: 0.7512407499106651.


In [6]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import numpy as np
import optuna

def objective2(trial, X, y):
    n_estimators = trial.suggest_int("n_estimators", 100, 1000, step=100)

    depth_option = trial.suggest_categorical("depth_type", ["unlimited", "constrained"])
    max_depth = None if depth_option == "unlimited" else trial.suggest_int("max_depth", 5, 50)

    min_samples_split = trial.suggest_int("min_samples_split", 2, 50)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    # Crucial: Class balancing is heavily needed for the Elliptic dataset
    class_weight = trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample", None])

    model = build_random_forest(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight
    )

    tscv = TimeSeriesSplit(n_splits=5, max_train_size=None)

    scores = cross_val_score(
        model, X, y, cv=tscv, scoring="f1", n_jobs=-1
    )

    return np.mean(scores)

# --- Execution ---
study = optuna.create_study(direction="maximize", study_name="Random_Forest_Ultimate_Tuning")

study.optimize(
    lambda trial: objective2(trial, x_all_train, y_all_train),
    n_trials=200,
    show_progress_bar=True
)

print("\n--- Optimization Finished ---")
print(f"Best Trial Score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-06-15 22:37:10,572] A new study created in memory with name: Random_Forest_Ultimate_Tuning
Best trial: 0. Best value: 0.715554:   0%|          | 1/200 [00:45<2:32:29, 45.98s/it]

[I 2026-06-15 22:37:56,550] Trial 0 finished with value: 0.7155535791778502 and parameters: {'n_estimators': 500, 'depth_type': 'unlimited', 'min_samples_split': 24, 'min_samples_leaf': 20, 'max_features': None, 'class_weight': None}. Best is trial 0 with value: 0.7155535791778502.


Best trial: 0. Best value: 0.715554:   1%|          | 2/200 [01:31<2:30:12, 45.52s/it]

[I 2026-06-15 22:38:41,743] Trial 1 finished with value: 0.5986575608433853 and parameters: {'n_estimators': 600, 'depth_type': 'constrained', 'max_depth': 32, 'min_samples_split': 40, 'min_samples_leaf': 11, 'max_features': None, 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.7155535791778502.


Best trial: 2. Best value: 0.727318:   2%|▏         | 3/200 [01:58<2:01:56, 37.14s/it]

...

Best trial: 82. Best value: 0.791742:  48%|████▊     | 95/200 [15:13<18:28, 10.55s/it]

[I 2026-06-15 22:52:23,828] Trial 94 finished with value: 0.7912471747674592 and parameters: {'n_estimators': 800, 'depth_type': 'constrained', 'max_depth': 29, 'min_samples_split': 17, 'min_samples_leaf': 10, 'max_features': 'log2', 'class_weight': 'balanced_subsample'}. Best is trial 82 with value: 0.7917417077368106.


Best trial: 82. Best value: 0.791742:  48%|████▊     | 95/200 [15:18<16:54,  9.66s/it]


[W 2026-06-15 22:52:28,732] Trial 95 failed with parameters: {'n_estimators': 900, 'depth_type': 'constrained', 'max_depth': 29, 'min_samples_split': 18, 'min_samples_leaf': 10, 'max_features': 'log2', 'class_weight': 'balanced_subsample'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\pansm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\pansm\AppData\Local\Temp\ipykernel_12084\1035839464.py", line 39, in <lambda>
    lambda trial: objective2(trial, x_train, y_train),
                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\pansm\AppData\Local\Temp\ipykernel_12084\1035839464.py", line 29, in objective2
    scores = cross_val_score(
        model, X, y, cv=tscv, scoring="f1", n_jobs=-1
    )
  File "C:\Users\pansm\AppData\Local\Packages\Py

KeyboardInterrupt: 

In [ ]:
def objective3(trial, X, y):
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
    n_estimators = trial.suggest_int("n_estimators", 100, 1000, step=100)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    gamma = trial.suggest_float("gamma", 0.0, 5.0)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True)

    model = build_xgboost(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        gamma=gamma,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        scale_pos_weight=7.634893125361063
    )

    tscv = TimeSeriesSplit(n_splits=5)

    scores = cross_val_score(
        model,
        X,
        y,
        cv=tscv,
        scoring="f1",
        n_jobs=-1
    )

    return np.mean(scores[-3:])

study = optuna.create_study(direction="maximize", study_name="XGBoost_Opt")

study.optimize(
    lambda trial: objective3(trial, x_all_train, y_all_train),
    n_trials=200,
    show_progress_bar=True
)

print("\n--- Optimization Finished ---")
print(f"Best Trial Score: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")